# 02 · Adopted Code and API Entry Point

**This notebook uses the V3.1 retrieval and generation pipeline.** By default, it reproduces the saved experiment on 50 questions: keyword Hit@5 is 78%, semantic Hit@5 is 82%, and hybrid retrieval with LLM evidence selection achieves 98%. These are V3.1 results. The first three code cells do not call an API. The fourth cell supports a new paid API experiment, with configurable parameters, hidden API-key input, and downloadable results.

All code and data are embedded; no additional file upload is required. Notebook 02 includes the original source modules, and notebook 06 presents their complete source for reading. The full offline workflow has been executed and verified. The revised API entry point has passed syntax checks, but no new paid run has been performed through it.

**Instructions:** Run the first three code cells in order. For the cached demonstration, leave the fourth cell at `RUN_LIVE = False`. To generate new model results, change it to `True`. The default run processes 5 questions; set `N_QUESTIONS = 50` for the complete test. Enter your API key in the hidden input prompt that appears below. The regulatory evaluation date remains 2026-10-02; the corpus is not automatically updated.

In [ ]:
try:
    import numpy
except ImportError:
    raise RuntimeError('缺少NumPy，请新增单元运行 %pip install numpy 后重试。')
import base64, io, zipfile, sys, json, runpy
from pathlib import Path
WORKSPACE=Path('/content/consumer_lending_demo') if Path('/content').exists() else Path.cwd()/'work'/'consumer_lending_demo'
WORKSPACE.mkdir(parents=True,exist_ok=True)
EMBEDDED_FIXTURE='...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(EMBEDDED_FIXTURE))) as archive:
    for member in archive.infolist():
        target=(WORKSPACE/member.filename).resolve()
        if not target.is_relative_to(WORKSPACE.resolve()):raise ValueError('Unsafe archive path')
        if not target.exists():
            target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(archive.read(member))
print('Saved experiment fixture loaded; no API calls.')
print('Workspace:',WORKSPACE)


In [ ]:
# Recompute the actual 50-question semantic and keyword rankings.
# Model-selected rankings are measured from saved outputs, not rerun.
result=runpy.run_path(str(WORKSPACE/'src/reproduce.py'))
metrics=json.loads((WORKSPACE/'generated/retrieval_metrics.json').read_text(encoding='utf-8'))
for method,values in metrics['metrics'].items():
    print(method, values['any_gold_hits'], '/50', f"{values['any_gold_rate']:.0%}")


In [ ]:
# Show a genuine retrieval failure, a correct worked example, and source text.
case=json.loads((WORKSPACE/'evals/q12_retrieval_failure.json').read_text(encoding='utf-8'))
print('Q12:',case['question'])
print('Expected clause:',case['gold'])
print('Semantic rank:',case['semantic_gold_rank'],'score:',round(case['semantic_gold_score'],6))
print('Keyword rank:',case['lexical_gold_rank'])
print('Semantic top five:',[(x['id'],round(x['score'],6)) for x in case['semantic_top5']])
import csv
rows=list(csv.DictReader((WORKSPACE/'results/v3/answers_for_review.csv').open(encoding='utf-8-sig')))
example=next(r for r in rows if r['Q_ID']=='Q01')
corpus=json.loads((WORKSPACE/'data/corpus.json').read_text(encoding='utf-8'))
print('\nQ01:',example['Question']);print(example['Answer'])
source=next(c for c in corpus if c['chunk_id']=='D01_A08')
print('\nSource D01_A08:',source['original_text']);print('Source URL:',source['source_url'])


In [ ]:
# OPTIONAL NEW API RUN — 默认关闭，不会重复扣费。
RUN_LIVE = False
N_QUESTIONS = 5  # 全部测试改成50；演示建议先用5题。

if RUN_LIVE:
    if not isinstance(N_QUESTIONS,int) or not 1 <= N_QUESTIONS <= 50:
        raise ValueError('N_QUESTIONS must be an integer from 1 to 50')
    try:
        import requests
    except ImportError:
        raise RuntimeError('缺少requests，请新增单元运行 %pip install requests 后重试。')
    import os, getpass, importlib.util, datetime, uuid, shutil
    live_dir = WORKSPACE/'live_runs'/('run_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6])
    previous_dir = os.environ.get('CONSUMER_LENDING_RUN_DIR')
    os.environ['CONSUMER_LENDING_RUN_DIR'] = str(live_dir)
    api_key = getpass.getpass('OpenRouter key (hidden): ')
    if not api_key.strip():
        if previous_dir is None: os.environ.pop('CONSUMER_LENDING_RUN_DIR',None)
        else: os.environ['CONSUMER_LENDING_RUN_DIR'] = previous_dir
        raise ValueError('API key cannot be empty')
    session = requests.Session()
    pipeline = None
    try:
        spec = importlib.util.spec_from_file_location('consumer_lending_live_pipeline',WORKSPACE/'src/pipeline.py')
        pipeline = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(pipeline)
        pipeline.LIMIT = N_QUESTIONS
        pipeline.key = api_key.strip()
        pipeline.session = session
        live_data = {'chunks':json.loads((WORKSPACE/'data/corpus.json').read_text(encoding='utf-8')),
                     'questions':json.loads((WORKSPACE/'data/questions.json').read_text(encoding='utf-8'))}
        # Generator receives questions and corpus, never expected/gold IDs.
        pipeline.run(live_data)
        gold_after_run = json.loads((WORKSPACE/'evals/gold_labels.json').read_text(encoding='utf-8'))
        pipeline.evaluate_retrieval(gold_after_run)
        live_status = json.loads((live_dir/'status.json').read_text(encoding='utf-8'))
        print('NEW RUN STATUS:',live_status)
        print('Check any errors; new outputs do not inherit old answer scores.')
    finally:
        session.close()
        api_key = ''
        if pipeline is not None: pipeline.key = ''
        if previous_dir is None: os.environ.pop('CONSUMER_LENDING_RUN_DIR',None)
        else: os.environ['CONSUMER_LENDING_RUN_DIR'] = previous_dir
    result_zip = Path(shutil.make_archive(str(live_dir)+'_results','zip',live_dir))
    print('新结果ZIP:',result_zip)
    try:
        from google.colab import files
        files.download(str(result_zip))
    except ImportError:
        print('非Colab环境，请从上述路径取出ZIP。')
else:
    print('API运行关闭。前面展示的是V3.1已有实验复现，不是新的付费运行。')


## Verification Update — 2026-10-03

The AI-assisted review of all 50 gold mappings and their key source clauses is completed in `09_Gold_and_Key_Clause_Verification_V3.1.ipynb`. D07's clipped text was supplemented by an indexed government republication. D04_S08 has a substantive new/existing-business transcription error; D05_S06/S08 have abbreviated source references; D02_A70 has footer noise. Corrected source text and recommended labels are stored separately in notebook 09. Notebook 02 intentionally retains the historical experiment; original metrics are unchanged. Full independent human review of all gold labels has not been claimed. The revised report has been approved by the student and is included in this package. The student supplies the demonstration recording separately.
